# 01 — Data Preparation

Pair real/fake videos from FaceForensics++, split into train/val/test.

**Requires:** FF++ dataset (face-cropped mp4 files).

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter

## Configuration

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/FF_Data"

# Expected layout:
#   DATA_ROOT/
#     000.mp4, 001.mp4, ...              (real videos)
#     FaceSwap/000_001.mp4, ...
#     Face2Face/000_001.mp4, ...
#     FaceShifter/000_001.mp4, ...
#     Deepfakes/000_001.mp4, ...

MANIPULATION_TYPES = ("FaceSwap", "Face2Face", "FaceShifter", "Deepfakes")
TOTAL_PAIRS = 1000
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
SEED = 42

## Video Pairing

`equal_from_each` cycles through manipulation types so each type is equally represented. Fallback to any available manipulation if the preferred type is missing.

In [ ]:
def get_video_pairs(data_root, strategy="equal_from_each",
                    total_real=1000, prefer_manipulation="FaceSwap",
                    manipulation_order=MANIPULATION_TYPES):
    real_paths = sorted(glob.glob(os.path.join(data_root, "*.mp4")))
    fake_paths = glob.glob(os.path.join(data_root, "*/*.mp4"))

    print(f"Real videos found: {len(real_paths)}")
    print(f"Fake videos found: {len(fake_paths)}")

    pairs = []

    if strategy == "equal_from_each":
        selected = real_paths[:min(total_real, len(real_paths))]
        n_types = max(1, len(manipulation_order))

        for idx, real_path in enumerate(selected):
            real_id = os.path.basename(real_path).split(".")[0]
            manip = manipulation_order[idx % n_types]

            candidates = [f for f in fake_paths
                          if os.path.basename(f).startswith(real_id + "_")
                          and manip in f]
            if not candidates:
                fallback = [f for f in fake_paths
                            if os.path.basename(f).startswith(real_id + "_")]
                if fallback:
                    candidates = [sorted(fallback)[0]]
                else:
                    continue

            pairs.append((real_path, sorted(candidates)[0]))

    elif strategy == "fixed":
        for real_path in real_paths:
            real_id = os.path.basename(real_path).split(".")[0]
            candidates = [f for f in fake_paths
                          if os.path.basename(f).startswith(real_id + "_")]
            preferred = [f for f in candidates if prefer_manipulation in f]
            if preferred:
                pairs.append((real_path, sorted(preferred)[0]))
            elif candidates:
                pairs.append((real_path, sorted(candidates)[0]))

    elif strategy == "random":
        for real_path in real_paths:
            real_id = os.path.basename(real_path).split(".")[0]
            candidates = [f for f in fake_paths
                          if os.path.basename(f).startswith(real_id + "_")]
            if candidates:
                pairs.append((real_path, random.choice(candidates)))

    print(f"Total pairs: {len(pairs)}")
    return pairs

In [ ]:
video_pairs = get_video_pairs(DATA_ROOT, strategy="equal_from_each",
                             total_real=TOTAL_PAIRS)

## Train / Val / Test Split

70/15/15 split. Pairs stay together so identities don't leak across splits.

In [ ]:
def split_pairs(pairs, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=SEED):
    pairs = pairs.copy()
    random.seed(seed)
    np.random.seed(seed)
    random.shuffle(pairs)

    n = len(pairs)
    t1 = int(n * train_ratio)
    t2 = t1 + int(n * val_ratio)

    train, val, test = pairs[:t1], pairs[t1:t2], pairs[t2:]
    print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")
    return train, val, test

train_pairs, val_pairs, test_pairs = split_pairs(video_pairs)

## Distribution Check

In [ ]:
def manipulation_stats(pairs, name=""):
    types = [os.path.basename(os.path.dirname(f)) for _, f in pairs]
    counts = Counter(types)
    print(f"\n{name} manipulation distribution:")
    for t, c in sorted(counts.items()):
        print(f"  {t:<14} {c:>4}")

manipulation_stats(train_pairs, "Train")
manipulation_stats(val_pairs, "Validation")
manipulation_stats(test_pairs, "Test")

In [ ]:
def video_length_distribution(pairs, sample=50):
    lengths = []
    for real_path, _ in pairs[:sample]:
        cap = cv2.VideoCapture(real_path)
        lengths.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        cap.release()

    plt.figure(figsize=(8, 3))
    plt.hist(lengths, bins=15, edgecolor="black", alpha=0.7)
    plt.xlabel("Frame count"); plt.ylabel("Videos")
    plt.title("Frame Count Distribution (sampled)")
    plt.tight_layout(); plt.show()

video_length_distribution(train_pairs)

In [ ]:
print("First 10 pairs:")
for r, f in video_pairs[:10]:
    print(f"  {os.path.basename(r):>10}  ->  {os.path.basename(f)}")